# Part 3 — The point cloud upsampler (super resolution), local GPU
Sparse S3DIS patch (256 points) → ×4 denser patch (1024 points), trained on (sparse, dense) pairs from the **training areas only**.

Model (pure PyTorch, ~160 k parameters): dense EdgeConv feature extractor → NodeShuffle-style feature expansion → coordinate regression (residual on the repeated input points) → one EdgeConv refinement stage.
Loss: `1000·Chamfer + 0.1·repulsion + 0.1·uniformity` (repulsion = PU-Net; uniformity = a **simplified** k-NN-distance variation proxy, not PU-GAN's disk-based loss; say so in the report).

**Prerequisite:** Part 1 data in `workspace/data/`. **Runtime:** building patch banks is CPU-bound (15–40 min); training takes minutes on a GPU.
Banks and trained upsamplers that already exist are skipped automatically (`--force` retrains one).

## 0. Setup and data check

In [ ]:
import sys
from pathlib import Path
PROJECT = Path.cwd().resolve()
if not (PROJECT / "pcseg").is_dir():          # notebook opened from the notebooks/ folder
    PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT))

import json, os, shutil
import numpy as np
import torch
from pcseg import settings, run, env_setup, results
settings.ensure_dirs()
D, S25, S12 = str(settings.DENSE), str(settings.SP25), str(settings.SP12)

g = settings.gpu_info()
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| gpu:", f"{g[0]} ({g[1]:.0f} GB)" if g else "NONE")
print("workspace:", settings.ROOT)
print("data     :", settings.DATA)
print(f"free disk: {shutil.disk_usage(settings.ROOT).free / 2**30:.0f} GB")

BANKS, UPS = str(settings.BANKS), str(settings.UPSAMPLER)
DENSE, SP25 = D, S25
ALL5 = ['Area_1', 'Area_2', 'Area_3', 'Area_4', 'Area_6']
from pcseg.pu_data import build_patch_bank

def train_up(out, train_bank, val_bank, *extra, tail=None, epochs=30, batch=32):
    """Train one upsampler (skipped when <out>/log.json exists). Runs pcseg/pu_train.py in a subprocess."""
    run.py_module('pcseg.pu_train', '--train-bank', train_bank, '--val-bank', val_bank, '--out', f'{UPS}/{out}',
                  '--epochs', epochs, '--batch', batch, *extra, tail=tail)

In [ ]:
from pcseg import data_check
assert data_check.check(), "fix the data problems listed above (README, step 3) before continuing"

## 1. Patch banks
- **Development**: train on Areas 1–4, validate on Area 6 (use this to tune loss weights and architecture).
- **Final** (section 4): train on all five development areas; Area 5 is never touched.
Each bank stores normalised (centre + unit-sphere) pairs: 256 sparse points ↔ the 1024 dense points of the same neighbourhood.
Do not stop bank building midway: an interrupted bank restarts from the beginning.

In [ ]:
build_patch_bank(DENSE, SP25, ['Area_1', 'Area_2', 'Area_3', 'Area_4'], f'{BANKS}/train_dev_x4.npz', keep_ratio=0.25, up_ratio=4, patches_per_room=100)
build_patch_bank(DENSE, SP25, ['Area_6'], f'{BANKS}/val_x4.npz', keep_ratio=0.25, up_ratio=4, patches_per_room=40, seed=1)

## 2. Train the development upsampler and compare with naive interpolation

In [ ]:
assert torch.cuda.is_available(), "No GPU visible to PyTorch: see README, Troubleshooting"
train_up('dev_x4', f'{BANKS}/train_dev_x4.npz', f'{BANKS}/val_x4.npz')

In [ ]:
import matplotlib.pyplot as plt
L = json.load(open(f'{UPS}/dev_x4/log.json'))
ep = [r['epoch'] for r in L['log']]
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
for a, k, t in zip(ax, ['val_cd', 'val_hd', 'val_f'], ['Chamfer (x1e3, lower is better)', 'Hausdorff (x1e3)', 'F-score@0.02 (higher is better)']):
    a.plot(ep, [r[k] for r in L['log']], label='upsampler')
    a.axhline(L['baseline_val'][k[4:]], color='r', ls='--', label='naive interpolation')
    a.set_title(t); a.set_xlabel('epoch')
ax[0].legend(); plt.tight_layout(); plt.savefig(settings.FIGURES / 'upsampler_dev_curves.png', dpi=130); plt.show()
print('final val:', {k: round(v, 3) for k, v in L['log'][-1].items() if k.startswith('val_')})
print('interp   :', {k: round(v, 3) for k, v in L['baseline_val'].items()})

## 3. Loss ablation (development setting)
Same data and schedule, only the loss changes. Report validation CD / HD / F-score / uniformity.

In [ ]:
assert torch.cuda.is_available(), "No GPU visible to PyTorch: see README, Troubleshooting"
variants = {'cd_only': ['--w-rep', '0', '--w-uni', '0'], 'cd_rep': ['--w-uni', '0'], 'cd_rep_uni': []}
for name, flags in variants.items():
    train_up(f'abl_{name}', f'{BANKS}/train_dev_x4.npz', f'{BANKS}/val_x4.npz', *flags, tail=2)

rows = []
for name in variants:
    L = json.load(open(f'{UPS}/abl_{name}/log.json'))
    rows.append((name, *[round(L['log'][-1][k], 3) for k in ['val_cd', 'val_hd', 'val_f', 'val_uni']]))
rows.append(('interpolation', *[round(L['baseline_val'][k], 3) for k in ['cd', 'hd', 'f', 'uni']]))
print('| loss | CD x1e3 | HD x1e3 | F@0.02 | uniformity CV |\n|---|---|---|---|---|')
for r in rows:
    print('| ' + ' | '.join(str(x) for x in r) + ' |')

## 4. Final upsampler for Area 5 (trained on all five development areas)
Validation patches come from Area 6 which is now part of training, so they are only a training monitor: use the **last** checkpoint, no selection.

In [ ]:
assert torch.cuda.is_available(), "No GPU visible to PyTorch: see README, Troubleshooting"
build_patch_bank(DENSE, SP25, ALL5, f'{BANKS}/train_full_x4.npz', keep_ratio=0.25, up_ratio=4, patches_per_room=100, seed=2)
train_up('full_x4', f'{BANKS}/train_full_x4.npz', f'{BANKS}/val_x4.npz', epochs=40)

## 5. (Optional but recommended) Cross-fit upsamplers for training PTv3 on upsampled scenes (row H)
An upsampler applied to the scenes it was trained on produces cleaner output than on unseen scenes. For row H we therefore upsample each training area with a model that **did not see it**:
fold A (trained on Areas 1,2,3) upsamples Areas 4,6; fold B (trained on Areas 4,6) upsamples Areas 1,2,3. Skip this section to use `full_x4` everywhere and state the limitation.

In [ ]:
assert torch.cuda.is_available(), "No GPU visible to PyTorch: see README, Troubleshooting"
build_patch_bank(DENSE, SP25, ['Area_1', 'Area_2', 'Area_3'], f'{BANKS}/train_foldA_x4.npz', keep_ratio=0.25, up_ratio=4, patches_per_room=100, seed=3)
build_patch_bank(DENSE, SP25, ['Area_4', 'Area_6'], f'{BANKS}/train_foldB_x4.npz', keep_ratio=0.25, up_ratio=4, patches_per_room=100, seed=4)
for fold in ['A', 'B']:
    train_up(f'fold{fold}_x4', f'{BANKS}/train_fold{fold}_x4.npz', f'{BANKS}/val_x4.npz', tail=1, epochs=40)

## 6. (Optional) Extra upsamplers for the ablations of Part 4
- **×2 upsampling** of the 25 % scan (target density 50 %): ground truth = random half of the dense neighbourhood.
- **12.5 % sparsity, ×8** upsampling back to full density (needs `workspace/data/s3dis_sparse12`).

In [ ]:
assert torch.cuda.is_available(), "No GPU visible to PyTorch: see README, Troubleshooting"
build_patch_bank(DENSE, SP25, ALL5, f'{BANKS}/train_full_x2.npz', keep_ratio=0.25, up_ratio=2, patches_per_room=100, seed=5)
build_patch_bank(DENSE, SP25, ['Area_6'], f'{BANKS}/val_x2.npz', keep_ratio=0.25, up_ratio=2, patches_per_room=40, seed=1)
train_up('full_x2', f'{BANKS}/train_full_x2.npz', f'{BANKS}/val_x2.npz', tail=1, epochs=40)

SP12 = str(settings.SP12)
if os.path.isdir(SP12):
    build_patch_bank(DENSE, SP12, ALL5, f'{BANKS}/train_full_x8_sp12.npz', keep_ratio=0.125, up_ratio=8, patches_per_room=100, seed=6)
    build_patch_bank(DENSE, SP12, ['Area_6'], f'{BANKS}/val_x8_sp12.npz', keep_ratio=0.125, up_ratio=8, patches_per_room=40, seed=1)
    train_up('full_x8_sp12', f'{BANKS}/train_full_x8_sp12.npz', f'{BANKS}/val_x8_sp12.npz', tail=1, epochs=40, batch=16)

## 7. Scene-level upsampler quality on Area 5 (metres) and cost
Sparse vs naive interpolation vs the learned upsampler, on the first 5 Area 5 rooms, measured against the original dense scans.
`chamfer_cm` = mean nearest-neighbour distance in both directions; `hd99_cm` = robust (99th-percentile) Hausdorff; F-score at 1 and 2 cm; uniformity = CV of the 8-NN distance.
How to read it: the sparse input consists of real points, so its precision is perfect and its Chamfer can look competitive; the fair comparison is learned versus interpolation, and the F-score and the segmentation rows decide the rest.

In [ ]:
assert torch.cuda.is_available(), "No GPU visible to PyTorch: see README, Troubleshooting"
from pcseg.common import load_room, room_dirs, write_json
from pcseg.pu_core import load_upsampler
from pcseg.pu_data import upsample_scene, interp_scene
from pcseg.up_metrics import scene_metrics, average, measure_upsampler_cost

model = load_upsampler(f'{UPS}/full_x4/last.pt', 'cuda')
acc = {'sparse': [], 'interpolation': [], 'upsampler': []}
for rd in room_dirs(SP25, 'Area_5')[:5]:
    sp = load_room(rd, ('coord', 'color'))
    dn = load_room(f'{DENSE}/Area_5/{rd.name}', ('coord',))['coord']
    acc['sparse'].append(scene_metrics(sp['coord'], dn))
    acc['interpolation'].append(scene_metrics(interp_scene(sp['coord'], sp['color'], 4)[0], dn))
    acc['upsampler'].append(scene_metrics(upsample_scene(model, sp['coord'], sp['color'], up_ratio=4, verbose=False)[0], dn))
summary = {k: average(v) for k, v in acc.items()}
for k, v in summary.items():
    print(f'{k:14s}', {a: round(b, 3) for a, b in v.items()})
cost = measure_upsampler_cost(model, SP25, 'Area_5', device='cuda', rooms=3)
print('cost:', cost)
write_json(settings.RESULTS / 'upsampler_scene_metrics.json', dict(metrics=summary, cost=cost))